## Micrograd Engine Benchmark

### 1. Imports

In [ ]:
import random
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles, make_friedman1
%matplotlib inline

# Unlike the other notebooks, this one benchmarks the finished engines instead
# of rebuilding them, so it imports them from src/ one level up.
sys.path.append("..")

from src.engines.tensor import Tensor
from src.nn.models import TensorMLP, ValueMLP
from src.nn.optim import SGD

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

### 2. Regression Dataset

In [ ]:
# make_friedman1 has a known non-linear ground truth:
#   y = 10*sin(pi*x0*x1) + 20*(x2-0.5)**2 + 10*x3 + 5*x4
# The remaining 5 features are noise, so the network has to learn what to ignore.
X, y = make_friedman1(n_samples=1000, n_features=10, noise=0.0, random_state=SEED)

# Every layer ends in tanh, so the network can only ever output values in
# (-1, 1) and the targets have to be rescaled into that range.
y = 2 * (y - y.min()) / (y.max() - y.min()) - 1

X.shape, y.shape, (y.min(), y.max())

In [ ]:
HIDDEN_SIZE = 16
LEARNING_RATE = 0.05
BATCH_SIZE = 32
EPOCHS = 10

### 3. Matching the Two Networks

In [ ]:
# TensorLayer is a plain affine map, while every ValueNeuron applies tanh to
# its own output. Applying tanh after each layer here makes the two networks
# compute the same function, which is what makes the timings comparable.
def tensor_forward(net, xb):
  out = Tensor(xb)
  for layer in net.layers:
    out = layer(out).tanh()
  return out

In [ ]:
def copy_weights(tensor_net, value_net):
  # TensorLayer holds one (input_size, output_size) matrix, ValueLayer holds one
  # neuron per output, so column j of that matrix is neuron j's weight vector.
  for tensor_layer, value_layer in zip(tensor_net.layers, value_net.layers):
    for j, neuron in enumerate(value_layer.neurons):
      for i, w in enumerate(neuron.w):
        w.data = float(tensor_layer.w.data[i, j])
      neuron.b.data = float(tensor_layer.b.data[j])

tensor_net = TensorMLP(10, [HIDDEN_SIZE, 1])
value_net = ValueMLP(10, [HIDDEN_SIZE, 1])

# Identical starting weights, so the two runs are the same experiment twice.
copy_weights(tensor_net, value_net)

sum(p.data.size for p in tensor_net.parameters())

In [ ]:
def batches(n, batch_size, rng):
  # Seeding the generator the same way for both runs feeds them the same
  # shuffled batches, so any difference in the loss curves is the engine's fault.
  idx = rng.permutation(n)
  for start in range(0, n - batch_size + 1, batch_size):
    yield idx[start:start + batch_size]

### 4. Training the Tensor Engine

In [ ]:
optimizer = SGD(tensor_net.parameters(), lr=LEARNING_RATE)
rng = np.random.default_rng(SEED)
tensor_losses = []

start = time.perf_counter()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for batch in batches(len(X), BATCH_SIZE, rng):
        # The whole batch goes through as one matmul per layer.
        preds = tensor_forward(tensor_net, X[batch])
        loss = ((preds - Tensor(y[batch].reshape(-1, 1)))**2).sum() / BATCH_SIZE

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += float(loss.data)
        n_batches += 1

    tensor_losses.append(epoch_loss / n_batches)

tensor_time = time.perf_counter() - start
print(f"Tensor engine: {tensor_time:.3f}s, final loss {tensor_losses[-1]:.4f}")

### 5. Training the Scalar Engine

In [ ]:
rng = np.random.default_rng(SEED)
value_losses = []

start = time.perf_counter()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    n_batches = 0

    for batch in batches(len(X), BATCH_SIZE, rng):
        # One Python-level forward pass per sample, and one graph node per
        # scalar operation inside it. That is the entire cost difference.
        loss = sum((value_net(list(X[i])) - float(y[i]))**2 for i in batch) / BATCH_SIZE

        # SGD.zero_grad() fills gradients with np.zeros_like, which assumes the
        # Tensor engine, so the scalar run uses Module.zero_grad and updates by hand.
        value_net.zero_grad()
        loss.backward()
        for p in value_net.parameters():
            p.data += -LEARNING_RATE * p.grad

        epoch_loss += loss.data
        n_batches += 1

    value_losses.append(epoch_loss / n_batches)

value_time = time.perf_counter() - start
print(f"Value engine: {value_time:.3f}s, final loss {value_losses[-1]:.4f}")

### 6. Speed Comparison

In [ ]:
speedup = value_time / tensor_time
print(f"Value engine:  {value_time:8.3f}s")
print(f"Tensor engine: {tensor_time:8.3f}s")
print(f"Speedup:       {speedup:8.0f}x")

# Same weights and same batches, so agreeing loss curves are evidence that the
# vectorized gradients are the scalar gradients, only faster.
largest_gap = max(abs(v - t) for v, t in zip(value_losses, tensor_losses))
print(f"\nLargest per-epoch loss difference: {largest_gap:.2e}")

In [ ]:
fig, (ax_time, ax_loss) = plt.subplots(1, 2, figsize=(11, 4))

ax_time.bar(["Value\n(scalar)", "Tensor\n(NumPy)"], [value_time, tensor_time],
            color=["tab:red", "tab:blue"])
ax_time.set_yscale("log")  # linear axes would flatten the faster bar to nothing
ax_time.set_ylabel("training time (s, log scale)")
ax_time.set_title(f"{EPOCHS} epochs on 1,000 samples: {speedup:.0f}x")
for i, seconds in enumerate([value_time, tensor_time]):
    ax_time.text(i, seconds, f"{seconds:.3f}s", ha="center", va="bottom")

epochs = range(1, EPOCHS + 1)
ax_loss.plot(epochs, value_losses, "o-", color="tab:red", label="Value (scalar)")
ax_loss.plot(epochs, tensor_losses, "--", color="tab:blue", label="Tensor (NumPy)")
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("MSE loss")
ax_loss.set_title("The curves overlap exactly")
ax_loss.legend()

plt.tight_layout()
plt.show()

### 7. Non-linear Classification Dataset

In [ ]:
# Concentric circles are not linearly separable, so a boundary that gets them
# right can only come from the hidden layer's non-linearity.
X_circles, y_circles = make_circles(n_samples=1000, noise=0.15, factor=0.5, random_state=SEED)
y_circles = np.where(y_circles == 0, -1.0, 1.0)  # tanh saturates at -1 and 1

perm = np.random.default_rng(SEED).permutation(len(X_circles))
X_circles, y_circles = X_circles[perm], y_circles[perm]

X_train, y_train = X_circles[:800], y_circles[:800]
X_val, y_val = X_circles[800:], y_circles[800:]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="RdBu", edgecolors="k", linewidths=0.3)
ax.set_title(f"make_circles: {len(X_train)} train, {len(X_val)} validation")
plt.show()

### 8. Loss and Accuracy Curves

In [ ]:
def evaluate(net, xb, yb):
  """Loss and accuracy over a whole split, in one batched forward pass."""
  preds = tensor_forward(net, xb)
  loss = float((((preds - Tensor(yb.reshape(-1, 1)))**2).sum() / len(xb)).data)
  # tanh is symmetric around 0, so the sign of the output is the predicted class.
  accuracy = float(np.mean(np.sign(preds.data.ravel()) == yb))
  return loss, accuracy

In [ ]:
CLASSIFIER_EPOCHS = 100

classifier = TensorMLP(2, [HIDDEN_SIZE, 1])
optimizer = SGD(classifier.parameters(), lr=LEARNING_RATE)
rng = np.random.default_rng(SEED)
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(CLASSIFIER_EPOCHS):
    for batch in batches(len(X_train), BATCH_SIZE, rng):
        preds = tensor_forward(classifier, X_train[batch])
        loss = ((preds - Tensor(y_train[batch].reshape(-1, 1)))**2).sum() / BATCH_SIZE

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Measured after the epoch's updates, on both splits, so the validation
    # curve says whether the fit generalizes or just memorizes.
    train_loss, train_acc = evaluate(classifier, X_train, y_train)
    val_loss, val_acc = evaluate(classifier, X_val, y_val)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

print(f"train: loss {history['train_loss'][-1]:.4f}, accuracy {history['train_acc'][-1]:.3f}")
print(f"val:   loss {history['val_loss'][-1]:.4f}, accuracy {history['val_acc'][-1]:.3f}")

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
epochs = range(1, CLASSIFIER_EPOCHS + 1)

ax_loss.plot(epochs, history["train_loss"], label="train")
ax_loss.plot(epochs, history["val_loss"], label="validation")
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("MSE loss")
ax_loss.set_title("Loss")
ax_loss.legend()

ax_acc.plot(epochs, history["train_acc"], label="train")
ax_acc.plot(epochs, history["val_acc"], label="validation")
ax_acc.set_xlabel("epoch")
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy")
ax_acc.legend()

plt.tight_layout()
plt.show()

### 9. Decision Boundary

In [ ]:
# The grid is just another batch, so the whole plane goes through the network
# in a single forward pass.
padding = 0.5
x_min, x_max = X_circles[:, 0].min() - padding, X_circles[:, 0].max() + padding
y_min, y_max = X_circles[:, 1].min() - padding, X_circles[:, 1].max() + padding
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

grid_preds = tensor_forward(classifier, np.c_[xx.ravel(), yy.ravel()])
zz = grid_preds.data.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 6))
ax.contourf(xx, yy, zz, levels=np.linspace(-1, 1, 21), cmap="RdBu", alpha=0.7)
ax.contour(xx, yy, zz, levels=[0.0], colors="k", linewidths=1.5)  # where the class flips
ax.scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap="RdBu", edgecolors="k", linewidths=0.5)
ax.set_title(f"Learned decision boundary (validation accuracy {history['val_acc'][-1]:.1%})")
plt.show()